In [11]:
MAX_VOCAB_SIZE = 20000   # or any number > max word index
MAX_SEQ_LEN = 256     # or 500, just be consistent


In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from pathlib import Path
import joblib

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.12.0


In [13]:
# Use the same raw or processed file as in Colab
DATA_PATH = Path("data/raw/fake_job_postings.csv")  # adjust if needed
df = pd.read_csv(DATA_PATH)

print("Dataset:", df.shape)
print("Fraud rate:", df["fraudulent"].mean() * 100)


Dataset: (17880, 18)
Fraud rate: 4.8434004474272925


In [14]:
def combine_text_fields(row, fields=(
    'title', 'location', 'company_profile', 'description',
    'requirements', 'benefits', 'required_experience',
    'required_education', 'industry', 'function',
)):
    parts = []
    for f in fields:
        if pd.notna(row[f]):
            parts.append(str(row[f]))
    return " ".join(parts) if parts else "unknown job"

print("Combining text fields...")
df["combined_text"] = df.apply(combine_text_fields, axis=1)
print("Done.")


Combining text fields...
Done.


In [15]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

print("Preprocessing text...")
df["text_processed"] = df["combined_text"].apply(preprocess_text)
print("Done.")


Preprocessing text...
Done.


In [24]:
X_text = df["text_processed"].values
y = df["fraudulent"].values.astype("int32")

# First: train vs temp (val+test)
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text,
    y,
    test_size=0.30,        # 70% train, 30% temp
    random_state=42,
    stratify=y,
)

# Second: split temp into val and test (15% + 15%)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print(len(X_train_text), len(X_val_text), len(X_test_text))


12516 2682 2682


In [25]:
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_SEQ_LEN, padding="post", truncating="post")


In [26]:
from tensorflow.keras.optimizers import Adam

print("\nBuilding LSTM model...")

EMBEDDING_DIM = 64
LSTM_UNITS    = 64
DENSE_UNITS   = 32


from tensorflow.keras.layers import LSTM, Bidirectional

model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQ_LEN,
    ),
    Bidirectional(LSTM(
        LSTM_UNITS,
        dropout=0.3,
        recurrent_dropout=0.3,
    )),
    Dropout(0.3),
    Dense(DENSE_UNITS, activation="relu"),
    Dense(1, activation="sigmoid"),
])

optimizer = Adam(learning_rate=5e-4)  # 0.0005

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizer,
    metrics=["accuracy"],
)

model.summary()


Building LSTM model...
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 256, 64)           1280000   
                                                                 
 bidirectional_2 (Bidirectio  (None, 128)              66048     
 nal)                                                            
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_4 (Dense)             (None, 32)                4128      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 1,350,209
Trainable params: 1,350,209
Non-trainable params: 0
____________________

In [27]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=8,
    batch_size=256,   # or 256
    callbacks=[early_stop],
    verbose=1,
)



Epoch 1/8
49/49 [==============================] - 165s 3s/step - loss: 0.3740 - accuracy: 0.9451 - val_loss: 0.1879 - val_accuracy: 0.9515
Epoch 2/8
49/49 [==============================] - 169s 3s/step - loss: 0.1588 - accuracy: 0.9517 - val_loss: 0.0831 - val_accuracy: 0.9556
Epoch 3/8
49/49 [==============================] - 177s 4s/step - loss: 0.0647 - accuracy: 0.9793 - val_loss: 0.0421 - val_accuracy: 0.9858
Epoch 4/8
49/49 [==============================] - 185s 4s/step - loss: 0.0303 - accuracy: 0.9899 - val_loss: 0.0436 - val_accuracy: 0.9877
Epoch 5/8
49/49 [==============================] - 184s 4s/step - loss: 0.0166 - accuracy: 0.9948 - val_loss: 0.0384 - val_accuracy: 0.9884
Epoch 6/8
49/49 [==============================] - 181s 4s/step - loss: 0.0097 - accuracy: 0.9977 - val_loss: 0.0425 - val_accuracy: 0.9888
Epoch 7/8
49/49 [==============================] - 194s 4s/step - loss: 0.0053 - accuracy: 0.9989 - val_loss: 0.0491 - val_accuracy: 0.9881
Epoch 8/8
49/49 [===

In [28]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"Validation accuracy: {val_acc:.4f}")


Validation accuracy: 0.9884


In [29]:
y_test_pred_proba = model.predict(X_test_pad).ravel()
y_test_pred = (y_test_pred_proba >= 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=4))

84/84 [==============================] - 3s 31ms/step
[[2522   30]
 [  20  110]]
              precision    recall  f1-score   support

           0     0.9921    0.9882    0.9902      2552
           1     0.7857    0.8462    0.8148       130

    accuracy                         0.9814      2682
   macro avg     0.8889    0.9172    0.9025      2682
weighted avg     0.9821    0.9814    0.9817      2682



In [30]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(tokenizer, MODELS_DIR / "tokenizer.pkl")
model.save(MODELS_DIR / "lstm_model.h5")

print("Saved tokenizer.pkl and lstm_model.h5")


Saved tokenizer.pkl and lstm_model.h5
